# Geographic Revenue & Delivery Risk Patterns

**Business question:** How do revenue concentration and delivery risk correlate across customer states, and which states represent the highest compound operational risk?

**Decisions supported:**
- Geographic resource allocation for logistics and support
- State-level SLA target differentiation
- Regional carrier contract prioritisation


## Data Sources

| Query | Description | Grain |
|---|---|---|
| Q3 of NB-01 | GMV, unique customers, total orders by state | One row per customer state |
| Q2 of NB-03 | Delay rate, delayed orders, total deliveries by state | One row per customer state |

**Cross-join:** The two state-grain tables are merged in Python on `customer_state`. No new SQL is introduced — this is a presentation-layer join of two existing aggregations.

**Derived columns (Python only, no SQL modification):**
- `gmv_share_pct`: state GMV as % of platform total
- `delay_tier`: above/below median delay rate
- `gmv_tier`: above/below median GMV
- `quadrant`: 2×2 composite (GMV tier × delay tier)
- `delay_per_1k`: delayed orders per 1,000 deliveries (normalised risk)


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

_REPO_ROOT = Path().resolve().parents[1]
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from analysis.utils.db import get_connection
from analysis.utils.sql_loader import get_sql_path, load_queries
from analysis.utils.plotting import apply_style, save_figure

apply_style()

# =============================================================================
# Notebook 06 — Geographic Patterns
# Setup: load state-grain revenue and SLA data, then cross-join in Python
# =============================================================================

# ---------------------------------------------------------------------------
# Load state-grain revenue data (Q3 of the revenue analysis)
# ---------------------------------------------------------------------------
sql_rev  = get_sql_path("sql/analysis/01_revenue_and_aov_behavior.sql")
q_states = load_queries(sql_rev)[2]   # Q3: GMV by state

# Load state-grain SLA data (Q2 of the SLA analysis)
sql_sla   = get_sql_path("sql/analysis/03_delivery_sla_performance.sql")
q_sla_agg = load_queries(sql_sla)[0]  # Q1: national totals
q_sla_geo = load_queries(sql_sla)[1]  # Q2: delay rate by state

# Load state-grain review score data (derived from review + delivery join)
sql_rev4  = get_sql_path("sql/analysis/04_review_score_drivers.sql")
q_del_score = load_queries(sql_rev4)[1]   # Q2: avg score by delivery status

with get_connection() as conn:
    df_gmv_state   = pd.read_sql(q_states,    conn)
    df_sla_summary = pd.read_sql(q_sla_agg,   conn)
    df_delay_state = pd.read_sql(q_sla_geo,   conn)

# ---------------------------------------------------------------------------
# Cross-join: merge revenue and SLA state-grain tables in Python
# SQL already computed both at state grain — this is presentation-layer join
# ---------------------------------------------------------------------------
df_geo = pd.merge(
    df_gmv_state,
    df_delay_state,
    on="customer_state",
    how="inner",
)

# Compute GMV share
total_gmv = df_geo["gmv"].sum()
df_geo["gmv_share_pct"] = (df_geo["gmv"] / total_gmv * 100).round(2)

# Classification: high delay = above median delayed_rate_pct
median_delay = df_geo["delayed_rate_pct"].median()
median_gmv   = df_geo["gmv"].median()
df_geo["delay_tier"]   = df_geo["delayed_rate_pct"].apply(
    lambda r: "High Delay"  if r > median_delay else "Low Delay"
)
df_geo["gmv_tier"] = df_geo["gmv"].apply(
    lambda g: "High GMV" if g > median_gmv else "Low GMV"
)
df_geo["quadrant"] = df_geo["gmv_tier"] + " / " + df_geo["delay_tier"]

# ---------------------------------------------------------------------------
# Validation checks
# ---------------------------------------------------------------------------
geo_checks = [
    ("Merged rows > 0",             len(df_geo) > 0),
    ("No null customer_state",      df_geo["customer_state"].notna().all()),
    ("No null gmv",                 df_geo["gmv"].notna().all()),
    ("No null delayed_rate_pct",    df_geo["delayed_rate_pct"].notna().all()),
    ("GMV values >= 0",             (df_geo["gmv"] >= 0).all()),
    ("Delay rate 0-100",            df_geo["delayed_rate_pct"].between(0, 100).all()),
]

print("Notebook 06 — Geographic Data Validation")
print("=" * 50)
for label, passed in geo_checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}]  {label}")

print(f"\nStates in merged dataset: {len(df_geo)}")
display(df_geo.sort_values("gmv", ascending=False).head(10))


## Analytical Methodology

**Methods applied:**
- **Gradient bar chart** (panel A): all-state GMV ranking with a single-hue gradient encoding magnitude — avoids false categorical distinction while preserving rank order.
- **Colour-coded bar chart** (panel B): delay rate coloured red/green relative to the national median. Median is more robust than mean for threshold benchmarking in right-skewed distributions.
- **Normalised horizontal bar** (panel C): delayed orders per 1,000 deliveries — this rate is independent of state size and more comparable across states of different volume.
- **2×2 scatter quadrant** (panel D): the most analytically dense panel. Four combinations of GMV tier and delay tier define four strategic situations requiring different interventions. State labels annotate the most important outliers.
- **Horizontal bar** (panel E): unique customer count by state — confirms whether GMV concentration and customer concentration are geographically aligned.

**Why this method:** Geographic analysis requires simultaneous evaluation of absolute scale (GMV, order counts) and relative risk (delay rate, normalised exposure). No single chart captures all dimensions — the quadrant scatter is the synthesis, with the other panels providing supporting detail.


In [ ]:
# =============================================================================
# Dashboard 06 — Geographic Revenue & Delivery Risk Patterns
# =============================================================================
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    "Olist Geographic Revenue & Delivery Risk Dashboard",
    fontsize=16, fontweight="bold", y=0.99,
)

df_sorted_gmv   = df_geo.sort_values("gmv", ascending=False)
df_sorted_delay = df_geo.sort_values("delayed_rate_pct", ascending=False)

# ---------------------------------------------------------------------------
# Panel A (top, wide): GMV bar sorted descending — all states
# ---------------------------------------------------------------------------
ax_gmv = fig.add_subplot(3, 2, (1, 2))

norm = plt.Normalize(df_sorted_gmv["gmv"].min(), df_sorted_gmv["gmv"].max())
bar_colors_gmv = plt.cm.Blues(norm(df_sorted_gmv["gmv"]) * 0.65 + 0.25)

ax_gmv.bar(
    df_sorted_gmv["customer_state"],
    df_sorted_gmv["gmv"],
    color=bar_colors_gmv,
)

# Annotate GMV share on top 3
for _, row in df_sorted_gmv.head(3).iterrows():
    ax_gmv.annotate(
        f"{row['gmv_share_pct']:.1f}% of GMV",
        xy=(row["customer_state"], row["gmv"]),
        xytext=(0, 6), textcoords="offset points",
        ha="center", fontsize=8, color="#1565C0",
    )

ax_gmv.set_title("A  |  Total GMV by State (all delivered orders)", loc="left", pad=8)
ax_gmv.set_xlabel("Customer State")
ax_gmv.set_ylabel("GMV (BRL)")
ax_gmv.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax_gmv.grid(True, axis="y", linestyle="--", alpha=0.5)
ax_gmv.set_axisbelow(True)

# ---------------------------------------------------------------------------
# Panel B (middle-left): Delay rate by state — all states sorted
# ---------------------------------------------------------------------------
ax_delay = fig.add_subplot(3, 2, 3)

bar_cols_delay = [
    "#F44336" if r > median_delay else "#4CAF50"
    for r in df_sorted_delay["delayed_rate_pct"]
]
ax_delay.bar(
    df_sorted_delay["customer_state"],
    df_sorted_delay["delayed_rate_pct"],
    color=bar_cols_delay, width=0.7,
)
ax_delay.axhline(
    median_delay, color="#FF9800", linewidth=1.4, linestyle="--",
    label=f"Median: {median_delay:.1f}%",
)
ax_delay.set_title("B  |  Delay Rate by State  (red = above median)", loc="left", pad=8)
ax_delay.set_xlabel("State")
ax_delay.set_ylabel("Delayed Orders (%)")
ax_delay.legend(fontsize=9)
ax_delay.grid(True, axis="y", linestyle="--", alpha=0.5)
ax_delay.set_axisbelow(True)

# ---------------------------------------------------------------------------
# Panel C (middle-right): Delayed orders per 1,000 deliveries — normalised risk
# ---------------------------------------------------------------------------
ax_norm = fig.add_subplot(3, 2, 4)

df_geo["delay_per_1k"] = (
    df_geo["delayed_orders"] / df_geo["total_deliveries"] * 1000
).round(1)
df_norm_sorted = df_geo.sort_values("delay_per_1k", ascending=True)

ax_norm.barh(
    df_norm_sorted["customer_state"],
    df_norm_sorted["delay_per_1k"],
    color="#FF9800",
)
ax_norm.set_title("C  |  Delayed Orders per 1,000 Deliveries", loc="left", pad=8)
ax_norm.set_xlabel("Delayed Orders / 1,000")
ax_norm.grid(True, axis="x", linestyle="--", alpha=0.5)
ax_norm.set_axisbelow(True)

# ---------------------------------------------------------------------------
# Panel D (bottom-left): 2×2 Risk quadrant — GMV vs. Delay Rate
# ---------------------------------------------------------------------------
ax_quad = fig.add_subplot(3, 2, 5)

quadrant_colors = {
    "High GMV / High Delay": "#F44336",
    "High GMV / Low Delay":  "#4CAF50",
    "Low GMV / High Delay":  "#FF9800",
    "Low GMV / Low Delay":   "#90A4AE",
}
for q, grp in df_geo.groupby("quadrant"):
    ax_quad.scatter(
        grp["gmv"] / 1e6,
        grp["delayed_rate_pct"],
        label=q,
        color=quadrant_colors.get(q, "#607D8B"),
        s=70, zorder=3,
    )

# Label top-risk and top-value states
for _, row in df_geo.nlargest(3, "gmv").iterrows():
    ax_quad.annotate(row["customer_state"],
                     xy=(row["gmv"] / 1e6, row["delayed_rate_pct"]),
                     xytext=(4, 2), textcoords="offset points", fontsize=8)
for _, row in df_geo.nlargest(2, "delayed_rate_pct").iterrows():
    ax_quad.annotate(row["customer_state"],
                     xy=(row["gmv"] / 1e6, row["delayed_rate_pct"]),
                     xytext=(4, 2), textcoords="offset points", fontsize=8)

ax_quad.axhline(median_delay,  color="#FF9800", linewidth=1, linestyle="--", alpha=0.6)
ax_quad.axvline(median_gmv / 1e6, color="#2196F3", linewidth=1, linestyle="--", alpha=0.6)
ax_quad.set_title("D  |  GMV vs. Delay Rate Risk Quadrant", loc="left", pad=8)
ax_quad.set_xlabel("GMV (M BRL)")
ax_quad.set_ylabel("Delayed Rate (%)")
ax_quad.legend(fontsize=8, loc="upper right")
ax_quad.grid(True, linestyle="--", alpha=0.4)

# ---------------------------------------------------------------------------
# Panel E (bottom-right): Unique customers per state (top 10)
# ---------------------------------------------------------------------------
ax_cust = fig.add_subplot(3, 2, 6)

df_top_cust = df_geo.sort_values("unique_customers", ascending=True).tail(10)
ax_cust.barh(
    df_top_cust["customer_state"],
    df_top_cust["unique_customers"],
    color="#9C27B0",
)
ax_cust.set_title("E  |  Top 10 States by Unique Customer Count", loc="left", pad=8)
ax_cust.set_xlabel("Unique Customers")
ax_cust.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax_cust.grid(True, axis="x", linestyle="--", alpha=0.5)
ax_cust.set_axisbelow(True)

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_figure(fig, "06_geographic_dashboard.png")
plt.show()


# Geographic Revenue & Delivery Risk Patterns — Conclusions

---

## Key Findings
- The top-ranking high-GMV states independently account for the majority of the platform's operational logistics volume.
- A quadrant cross-analysis confirms that the highest-GMV state also operates with a delivery delay rate above the national median.
- Unique customer density mirrors GMV distribution perfectly, with no evidence of high-customer, low-spend outlier states.
- Several low-visibility mid-tier states experience disproportionately high normalised delay risk (delays per 1,000 deliveries).
- Logistics performance failures are structurally embedded in the regions generating the highest financial returns.

## Business Implications
- Operational failures (e.g., carrier outages or support bottlenecks) in high-GMV states simultaneously trigger compounding revenue, retention, and satisfaction crises.
- Allocating logistics remediation resources equally across all regions systematically under-serves the highest-value customer bases.
- States exhibiting high normalised delay rates represent hidden churn engines operating beneath the visibility of top-line volume dashboards.

## Actionable Recommendations
- Prioritize immediate service-level intervention for the specific high-GMV state currently operating above the median delay limit.
- Calculate and monitor a composite state-level operational risk index weighting GMV, absolute delay count, and delay rate.
- Adjust inventory positioning algorithms to explicitly favor fulfillment centers servicing high-priority, high-delay quadrants.
